In [ ]:
import pandas as pd
import pypsa
import yaml
import pycountry

### File paths

In [ ]:
production_fn = '../../resources/ironore-production.csv'
bus_locations_fn = '../../data/bus_locations.csv'

# Outputs
production_clustered_fn = '../../resources/ironore_production_clustered.csv'

### Get iron ore data

In [ ]:

# Read the CSV data
production = pd.read_csv(production_fn, index_col='ISO_A2')

In [ ]:
production

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config file
with open('../../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
regions = config['regions']

# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_2
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_2

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        # Some country names may have extra text (e.g., "Togo + Algeria"), handle them simply
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

# Map each row in production to its region
production['region'] = production.index.map(lambda iso: iso_to_region.get(iso, 'Other'))
production.sort_values(by='IronOreProductionGt', ascending=False)

### Aggregate iron ore production by region

In [ ]:
# Group by region and sum all numeric columns
production_by_region = production.groupby('region').sum(numeric_only=True)
production_by_region

### Save clustered iron ore production data

In [ ]:
production_by_region.to_csv(production_clustered_fn)

### Add Iron Ore to PyPSA network

In [ ]:
bus_locations = pd.read_csv(bus_locations_fn,header=0, index_col=0)

In [ ]:
bus_locations.rename(columns={'lat': "y", "long": "x"}, inplace=True)

In [ ]:
bus_locations.index = bus_locations.index + ' iron ore'

In [ ]:
bus_locations

In [ ]:
n = pypsa.Network()

In [ ]:
n.add(
    "Carrier",
    "ironore",
    nice_name="Iron Ore",
)


In [ ]:
n.madd(
    "Bus",
    names = production_by_region.index + " iron ore",
    x=bus_locations.x,
    y=bus_locations.y,
    carrier="ironore",
)

In [ ]:
n.buses

In [ ]:
hourly_output = production_by_region['IronOreProductionGt']/8760
hourly_output.index = production_by_region.index + " iron ore"
hourly_output.name = "region"

In [ ]:
n.madd(
    "Generator",
    names=production_by_region.index + " iron ore",
    bus=production_by_region.index + " iron ore",
    p_nom=hourly_output,
    p_nom_extendable=False,
    p_max_pu=1,
    carrier="ironore",
    capital_cost=10.0,
    marginal_cost=10.0,
)

### Testing

In [ ]:
n.add(
    "Load",
    name="testload",
    bus="Europe iron ore",
    p_set=0.001,
)

In [ ]:
n.optimize(solver_name="gurobi")

In [ ]:
n.statistics()